In [4]:
# Load the dataset

import pandas as pd

df=pd.read_csv("IMDB Dataset.csv")
print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [6]:
df["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [7]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
df["review_length"] = df["review"].apply(len)
df["review_length"].describe()

count    50000.000000
mean      1309.431020
std        989.728014
min         32.000000
25%        699.000000
50%        970.000000
75%       1590.250000
max      13704.000000
Name: review_length, dtype: float64

In [9]:
print(df["review"][0])
print(df["review"][67])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

In [10]:
import nltk
nltk.download("stopwords")
nltk.download("wordnet")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [13]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text=text.lower()
    text=re.sub(r"<.*?>","",text)
    text=text.translate(str.maketrans("","",string.punctuation))
    text=re.sub(r"\d+","",text)

    words=text.split()

    words=[
        lemmatizer.lemmatize(word) 
        for word in words 
        if word not in stop_words
    ]

    return " ".join(words)

In [14]:
df["review"].head().apply(clean_text)

0    one reviewer mentioned watching oz episode you...
1    wonderful little production filming technique ...
2    thought wonderful way spend time hot summer we...
3    basically there family little boy jake think t...
4    petter matteis love time money visually stunni...
Name: review, dtype: object

In [15]:
df["clean_review"] = df["review"].apply(clean_text)

In [16]:
df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

In [17]:
df.head()

,review,sentiment,review_length,clean_review
0,One of the other reviewers has mentioned that ...,1,1761,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,1,998,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,1,926,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,0,748,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",1,1317,petter matteis love time money visually stunni...


In [18]:
from sklearn.model_selection import train_test_split

X = df["clean_review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

(40000,)
(10000,)


In [19]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(
    num_words=10000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

In [20]:
print(len(tokenizer.word_index))

176479


In [22]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print(X_train.iloc[0])
print(X_train_seq[0])

caught little gem totally accident back revival theatre see two old silly scifi movie theatre packed full warning showed bunch scifi short spoof get u mood somewhat amusing came within second audience hysteric biggest laugh came showed princess laia huge cinnamon bun instead hair head look camera give grim smile nod made even funnier gotta see chewabacca played look like muppet extremely silly stupidbut couldnt stop laughing dialogue drowned laughter also know star war pretty well even funnierthey deliberately poke fun dialogue really work audience definite
[897, 46, 1128, 344, 1391, 62, 7615, 1294, 13, 38, 83, 548, 771, 2, 1294, 3020, 276, 1553, 1066, 611, 771, 223, 1924, 9, 79, 1024, 531, 994, 271, 624, 202, 164, 1, 943, 220, 271, 1066, 2414, 1, 507, 1, 1, 205, 992, 258, 43, 254, 60, 2962, 1420, 3622, 28, 10, 2488, 2929, 13, 1, 163, 43, 5, 4084, 458, 548, 1, 306, 417, 884, 282, 7708, 1990, 22, 33, 111, 184, 92, 18, 10, 1, 3841, 5353, 155, 282, 15, 55, 164, 3197]


In [23]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 200

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

print(X_train_pad.shape)
print(X_test_pad.shape)

(40000, 200)
(10000, 200)


In [24]:
import numpy as np

lengths = [len(seq) for seq in X_train_seq]
print(np.mean(lengths))

118.76205


In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(
        input_dim=10000,
        output_dim=128
    ),

    LSTM(64),

    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [30]:
model.build(input_shape=(None, 200))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,473 (5.07 MB)

 Trainable params: 1,329,473 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 40s 76ms/step - accuracy: 0.5073 - loss: 0.6934 - val_accuracy: 0.5130 - val_loss: 0.6930
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 70ms/step - accuracy: 0.5290 - loss: 0.6857 - val_accuracy: 0.5239 - val_loss: 0.6889
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 71ms/step - accuracy: 0.5463 - loss: 0.6525 - val_accuracy: 0.5437 - val_loss: 0.6757
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 70ms/step - accuracy: 0.6734 - loss: 0.5464 - val_accuracy: 0.8576 - val_loss: 0.3461
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 70ms/step - accuracy: 0.8995 - loss: 0.2737 - val_accuracy: 0.8726 - val_loss: 0.3421


In [32]:
test_loss, test_acc = model.evaluate(
    X_test_pad,
    y_test
)

print("Test Accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.8678 - loss: 0.3523
Test Accuracy: 0.871399998664856


In [33]:
model.save("models/lstm_model.keras")

In [34]:
predictions = model.predict(X_test_pad)
pred_labels = (predictions > 0.5).astype(int)

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step


In [35]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        pred_labels
    )
)

              precision    recall  f1-score   support

           0       0.89      0.84      0.87      5000
           1       0.85      0.90      0.88      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [36]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, pred_labels))

[[4212  788]
 [ 498 4502]]


In [37]:
from tensorflow.keras.layers import GRU

model = Sequential([
    Embedding(10000, 128),
    GRU(64),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [38]:
model.build(input_shape=(None, 200))
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,317,313 (5.03 MB)

 Trainable params: 1,317,313 (5.03 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
history_gru = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 37s 71ms/step - accuracy: 0.5004 - loss: 0.6935 - val_accuracy: 0.5148 - val_loss: 0.6916
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 34s 69ms/step - accuracy: 0.5392 - loss: 0.6796 - val_accuracy: 0.5497 - val_loss: 0.6816
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 34s 69ms/step - accuracy: 0.5957 - loss: 0.6571 - val_accuracy: 0.8388 - val_loss: 0.4010
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 35s 70ms/step - accuracy: 0.8941 - loss: 0.2817 - val_accuracy: 0.8824 - val_loss: 0.3017
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 37s 75ms/step - accuracy: 0.9470 - loss: 0.1617 - val_accuracy: 0.8724 - val_loss: 0.3203


In [40]:
test_loss, test_acc = model.evaluate(
    X_test_pad,
    y_test
)

print("Test Accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.8684 - loss: 0.3374
Test Accuracy: 0.8708000183105469


In [41]:
model.save("models/gru_model.keras")

In [42]:
predictions = model.predict(X_test_pad)
pred_labels = (predictions > 0.5).astype(int)

313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step


In [43]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        pred_labels
    )
)

              precision    recall  f1-score   support

           0       0.86      0.88      0.87      5000
           1       0.88      0.86      0.87      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [44]:
from tensorflow.keras.layers import  Bidirectional

bilstm_model = Sequential([
    Embedding(
        input_dim=10000,
        output_dim=128
    ),

    Bidirectional(
        LSTM(64)
    ),

    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

bilstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_model.build(input_shape=(None, 200))
bilstm_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,378,945 (5.26 MB)

 Trainable params: 1,378,945 (5.26 MB)

 Non-trainable params: 0 (0.00 B)

In [45]:
history_bilstm = bilstm_model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 61s 118ms/step - accuracy: 0.7344 - loss: 0.5121 - val_accuracy: 0.8767 - val_loss: 0.3028
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 55s 109ms/step - accuracy: 0.9033 - loss: 0.2606 - val_accuracy: 0.8759 - val_loss: 0.3062
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 54s 109ms/step - accuracy: 0.9251 - loss: 0.2035 - val_accuracy: 0.8555 - val_loss: 0.3793
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 53s 107ms/step - accuracy: 0.9391 - loss: 0.1688 - val_accuracy: 0.8746 - val_loss: 0.4080
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 55s 109ms/step - accuracy: 0.9595 - loss: 0.1172 - val_accuracy: 0.8741 - val_loss: 0.3892


In [46]:
test_loss, test_acc = bilstm_model.evaluate(
    X_test_pad,
    y_test
)

print("BiLSTM Test Accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.8694 - loss: 0.3999
BiLSTM Test Accuracy: 0.8719000220298767


In [47]:
from sklearn.metrics import classification_report

predictions = bilstm_model.predict(X_test_pad)

pred_labels = (predictions > 0.5).astype(int)

print(classification_report(y_test, pred_labels))

313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step
              precision    recall  f1-score   support

           0       0.87      0.88      0.87      5000
           1       0.88      0.86      0.87      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [49]:
bilstm_model.save("models/bilstm_model.keras")